# 05 — Finalize Dataset

**Goal:** Apply experience filter (min 5 fights), weight-class Z-score normalization, and produce the final modeling dataset. Includes a validation step to confirm normalization worked.

**Input:** `ufc_fighter_style_features.csv`, `ufc_fight_stats_cleaned.csv`  
**Output:** `ufc_modeling_data_final.csv`

In [1]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 05_finalize_dataset.ipynb | code cell index: 1
# Section (from markdown above): 05 — Finalize Dataset
# ------------------------------------------------------------------------
# Dependencies: see import statements below.
# Loads one or more CSV files (paths usually relative to notebooks/).
# Persists a DataFrame to CSV under data/processed or similar.
# Joins tables on fighter, fight, or event keys.
# Groups rows for aggregation (means, counts, etc.).
# ========================================================================
# INLINE_WORKFLOW_SUMMARY v1
# Workflow summary (how to read this cell):
#   • Input/output paths are configured at the top; adjust if your data live elsewhere.
#   • CSV reads use paths relative to the notebooks/ working directory.
#   • Processed outputs are written so downstream notebooks can load them.
#   • Merges link fighters, fights, styles, or odds on shared keys.
#   • Groupby operations aggregate fight or fighter statistics.

# --- Core pipeline load for this notebook ---
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt

# Config
INPUT_PATH = '../data/processed/ufc_fighter_style_features.csv'
CLEANED_FIGHTS_PATH = '../data/processed/ufc_fight_stats_cleaned.csv' # Needed for weight class context
OUTPUT_PATH = '../data/processed/ufc_modeling_data_final.csv'

# DECISIONS (Based on EDA results)
MIN_FIGHTS = 5 # Removes fighters with < 5 fights to reduce noise
NORMALIZE_WEIGHT = True # Z-scores stats relative to division average

def z_score_by_group(df, value_col, group_col):
    # Normalizes a column relative to its weight class average. Formula: (Value - Division_Mean) / Division_Std_Dev
    groups = df.groupby(group_col)[value_col]
    mean = groups.transform('mean')
    std = groups.transform('std').replace(0, 1) # Prevent division by zero
    return (df[value_col] - mean) / std

# Load data
if os.path.exists(INPUT_PATH) and os.path.exists(CLEANED_FIGHTS_PATH):
    df = pd.read_csv(INPUT_PATH)
    df_raw = pd.read_csv(CLEANED_FIGHTS_PATH)
    
    # Re-Attach Weight Class
    # We use the mode for each fighter to assign their primary division
    fighter_weights = df_raw.groupby('Fighter')['Weight_Class'].agg(
        lambda x: x.mode()[0] if len(x.mode()) > 0 else "Unknown"
    ).reset_index()
    
    df = df.merge(fighter_weights, on='Fighter')
    
    initial_count = len(df)
    
    # Apply Experience Filter
    df_clean = df[df['Total_Fights'] >= MIN_FIGHTS].copy()
    dropped_count = initial_count - len(df_clean)
    
    print(f"1. Experience Filter (Min {MIN_FIGHTS} fights):")
    print(f"   - Dropped {dropped_count} fighters")
    print(f"   - Remaining: {len(df_clean)} fighters")

    # Apply Weight Class Normalization (Z-Scoring)
    if NORMALIZE_WEIGHT:
        print("\n2. Applying Weight Class Normalization...")
        
        # Columns to normalize (Volume/Pace metrics that vary by weight)
        metrics_to_norm = [
            'Sig_Str_PM', 'Takedown_Att_PM', 'Sub_Att_PM', 
            'Control_Ratio'
        ]
        
        for col in metrics_to_norm:
            new_col_name = f"{col}_Z"
            df_clean[new_col_name] = z_score_by_group(df_clean, col, 'Weight_Class')
            
        print(f"   - Created normalized columns: {[c+'_Z' for c in metrics_to_norm]}")

    # Save
    df_clean.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSUCCESS! Final dataset saved to: {OUTPUT_PATH}")
    print("You are ready for style discovery (notebooks 08–10).")
    
else:
    print(f"Error: Missing input files at {INPUT_PATH} or {CLEANED_FIGHTS_PATH}")

1. Experience Filter (Min 5 fights):
   - Dropped 1410 fighters
   - Remaining: 1191 fighters

2. Applying Weight Class Normalization...
   - Created normalized columns: ['Sig_Str_PM_Z', 'Takedown_Att_PM_Z', 'Sub_Att_PM_Z', 'Control_Ratio_Z']

SUCCESS! Final dataset saved to: ../data/processed/ufc_modeling_data_final.csv
You are ready for Week 6: Clustering.


## Validation: Normalization Check

Confirm Z-scores are centered at 0 and weight classes are aligned.

In [ ]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 05_finalize_dataset.ipynb | code cell index: 3
# Section (from markdown above): Validation: Normalization Check
# ------------------------------------------------------------------------
# Dependencies: see import statements below.
# Loads one or more CSV files (paths usually relative to notebooks/).
# Builds matplotlib/seaborn figures.
# ========================================================================
# INLINE_WORKFLOW_SUMMARY v1
# Workflow summary (how to read this cell):
#   • Input/output paths are configured at the top; adjust if your data live elsewhere.
#   • CSV reads use paths relative to the notebooks/ working directory.

# Load the saved dataset and validate normalization
import seaborn as sns
import matplotlib.pyplot as plt

df_check = pd.read_csv(OUTPUT_PATH)

# 1. Z-score distributions should be centered at ~0
plt.figure(figsize=(10, 5))
sns.kdeplot(df_check['Sig_Str_PM_Z'], label='Striking Volume (Z)', fill=True)
sns.kdeplot(df_check['Takedown_Att_PM_Z'], label='Takedown Attempts (Z)', fill=True)
plt.title("Normalization check: distributions centered at 0")
plt.xlabel("Standard deviations from division average")
plt.legend()
plt.show()

# 2. Weight classes should align at Z=0 (Flyweight vs Heavyweight)
main_classes = ['Flyweight', 'Heavyweight']
df_subset = df_check[df_check['Weight_Class'].isin(main_classes)]
plt.figure(figsize=(8, 5))
sns.boxplot(x='Weight_Class', y='Sig_Str_PM_Z', data=df_subset, palette='coolwarm')
plt.title("Sanity check: Heavyweights and Flyweights aligned at Z=0")
plt.axhline(0, color='red', linestyle='--')
plt.ylabel("Z-Score (normalized activity)")
plt.show()

**Interpretation:**  
- **Left plot:** The KDE curves are centered near 0, showing Z-scores behave as intended. Striking volume and takedown attempts are normalized relative to each division’s average.  
- **Right plot:** Heavyweight and Flyweight boxplots align around Z=0, confirming that we’re comparing fighters to their *own* division rather than mixing divisions. This justifies using Z-scores for cross-division style clustering.

### Stars in the final modeling set (Z-scores)
Z-scores are **within weight class**. **Interpretation:** A negative `Sig_Str_PM_Z` is ‘below division average’, not ‘bad’—elites often win on efficiency and matchup.


In [ ]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 05_finalize_dataset.ipynb | code cell index: 6
# Section (from markdown above): Stars in the final modeling set (Z-scores)
# ------------------------------------------------------------------------
# Dependencies: see import statements below.
# Loads one or more CSV files (paths usually relative to notebooks/).
# ========================================================================
# INLINE_WORKFLOW_SUMMARY v1
# Workflow summary (how to read this cell):
#   • CSV reads use paths relative to the notebooks/ working directory.

import pandas as pd
stars = ['Jon Jones', 'Israel Adesanya', 'Khabib Nurmagomedov', 'Conor McGregor', 'Georges St-Pierre', 'Amanda Nunes', 'Anderson Silva', 'Max Holloway', 'Stipe Miocic']
m = pd.read_csv('../data/processed/ufc_modeling_data_final.csv')
zcols = [c for c in m.columns if c.endswith('_Z')]
sub = m[m['Fighter'].isin(stars)]
show = ['Fighter','Weight_Class','Total_Fights'] + zcols
print(sub[show].round(2).to_string(index=False))
